# Lab 2.4 · A reporting rate for districts without sensors

Day 2 · Session 4 (Under-reporting as measurement error) · 25 min.

Districts with sensors record every burst. In the 76 districts without sensors, a burst is recorded only when someone reports it. In 2024 Ouda Al-Shahwan's leak detection team listened to six districts without sensors for a season: of the bursts they found, 70% had been reported. You turn that audit into a prior on the share reported, add a reporting term to the model, and see what changes for the districts without sensors.

**AI policy.** The tutor Mirsad may explain; it writes no model code.

**Reference state.** `data/districts.csv` and `data/bursts.csv`, loaded fresh; the model of Lab 2.2 is available as `U.burst_model`, so Lab 2.2 need not be finished.

**Tasks**
- [ ] 1 · Reported bursts with and without sensors (4')
- [ ] 2 · The audit prior on the reporting share (5')
- [ ] 3 · Add the reporting term (7')
- [ ] 4 · Corrected rates for districts without sensors (6')
- [ ] 5 · What the data cannot tell apart (3')
- [ ] ✱ Stretch · Change the audit prior and watch the corrected rates follow it

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-401.git"
    if not Path("/content/ds-401").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-401"], check=True)
    os.chdir("/content/ds-401/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("yanabi.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from scipy import stats
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
import yanabi as U
print("mode:", U.MODE)

D = U.design()
md = U.model_data(D)
kind = np.array(D["kind"])
sens = D["sensors"]
true_rate = np.array([U.truth()["true_rate_per_km_year"][k] for k in D["ids"]])
print(f"districts without sensors: {(~sens).sum()} (rural {((~sens) & (kind == 'rural')).sum()})")

## 1 · Reported bursts with and without sensors (4')

✏️ **Exercise** · For rural districts only, return the reported bursts per km-year of districts with sensors and of districts without (total bursts over total km-years, a dict with keys `sensors` and `no_sensors`).

In [ ]:
def raw_rates():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    raw = raw_rates()
except NotImplementedError:
    raw = U.reference("d2_4", "raw")
U.keep("d2_4", "raw", raw)
print(f"rural districts, reported bursts per km-year: with sensors {raw['sensors']:.3f}, without {raw['no_sensors']:.3f}")
U.check(raw["no_sensors"] < raw["sensors"], "rural districts without sensors report fewer bursts per km-year")

The gap could be real (different pipes) or unrecorded bursts. The counts alone cannot say which: a district with rate λ and reporting share p produces exactly the same reported counts as one with rate λp and full reporting.

## 2 · The audit prior on the reporting share (5')

The audit found 28 of 40 bursts had been reported. A Beta(14, 6) prior carries the audit's 70% with the weight of about 20 bursts, half the audit's 40, because six districts in one season may not represent all 76.

✏️ **Exercise** · Return the prior mean and 90% interval of the reporting share under Beta(14, 6), and the same for Beta(28, 12), the audit taken at full weight.

In [ ]:
def audit_prior():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    ap = audit_prior()
except NotImplementedError:
    ap = U.reference("d2_4", "audit_prior")
U.keep("d2_4", "audit_prior", ap)
for k, (m, lo, hi) in ap.items():
    print(f"{k}: mean {m:.2f}, 90% interval {lo:.2f} to {hi:.2f}")
U.check(abs(ap["Beta(14, 6)"][0] - 0.70) < 1e-9, "both priors are centred on the audit's 70%")

## 3 · Add the reporting term (7')

Reported bursts ~ Poisson(km × rate × p_j), with p_j = 1 where there are sensors and p_j ~ Beta(14, 6) where there are none. On the log scale: add log p_j to the log mean.

✏️ **Exercise** · Complete `reporting_model`: the Lab 2.2 model (non-centred, material and age) plus the reporting term. Name the reporting site `p_report`.

In [ ]:
def reporting_model(d, log_km, X, sensors, J, y=None):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    numpyro.handlers.seed(reporting_model, 0)(**md)
    model = reporting_model
except NotImplementedError:
    U.safety("reporting_model")
    model = lambda **kw: U.burst_model(reporting=True, **kw)
_, rep = U.sample(model, y=D["y"], **md)
_, base = U.sample(U.burst_model, y=D["y"], **md)  # Lab 2.2's model, without the reporting term
dr = U.diagnose(rep, ["alpha", "beta", "tau"])
print({k: round(v, 3) for k, v in dr.items()})
U.check(U.trustworthy(dr), "the model with the reporting term can be read")

## 4 · Corrected rates for districts without sensors (6')

✏️ **Exercise** · Return a DataFrame by kind and sensors with the mean of log(estimate / true rate) for both models, and the mean width of the 90% interval of the rate relative to its median (`rel_width`) for both.

In [ ]:
def corrected():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    cor = corrected()
except NotImplementedError:
    cor = U.reference("d2_4", "corrected")
U.keep("d2_4", "corrected", cor)
print(cor.round(3).to_string())
rb = cor.loc[("rural", False)]
U.check(abs(rb[("bias", "reporting")]) < abs(rb[("bias", "base")]) - 0.15, "the reporting term removes most of the rural districts' bias")
ub = cor.loc[("urban", False)]
U.check(ub[("rel_width", "reporting")] > ub[("rel_width", "base")] + 0.05, "urban districts without sensors get wider intervals: the uncertainty about reporting is now in the answer")

**Look at** the widths. Urban and suburban districts without sensors had precise counts, so the reporting share is now the main uncertainty and their intervals widen. Rural districts' intervals were already wide from having few bursts; the correction moves them up more than it widens them.

## 5 · What the data cannot tell apart (3')

✏️ **Exercise** · For the districts without sensors, return the average ratio of the posterior standard deviation of p_report to its prior standard deviation (Beta(14, 6)). A ratio near 1 means the data taught the model almost nothing about reporting.

In [ ]:
def learned():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    ratio = learned()
except NotImplementedError:
    ratio = U.reference("d2_4", "learned")
U.keep("d2_4", "learned", ratio)
print(f"posterior sd of the reporting share over its prior sd: {ratio:.2f}")
U.check(ratio > 0.8, "the reporting share is carried by the audit prior, not by the counts")

The correction is only as good as the audit. Nouran's memo must say so, and a better audit (or the survey priced on Day 4) is the way to improve it.

## ✱ Stretch · Another audit prior

Refit with Beta(9, 1) (reporting is nearly complete) and with Beta(5, 5). How do the corrected rural rates follow the prior?

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ stretch not done")

In [ ]:
U.note("Checked: reported counts are now modelled as true counts thinned by a reporting share.",
       "Found: the correction comes from the audit prior; the counts cannot identify it.",
       "Carry forward: this model, with a negative binomial in place of the Poisson (Lab 3.1), is the one Day 4 decides with.")

## Checkpoint

1. Why can't the counts identify the reporting share by themselves?
   <details><summary>Answer</summary>Only the product of rate and reporting share reaches the counts; many pairs give the same product.</details>
2. Why did the intervals of the urban districts without sensors widen?
   <details><summary>Answer</summary>Their counts were precise; the uncertainty about how many bursts went unrecorded is now part of the posterior instead of being assumed away.</details>
3. Why use the audit at half weight?
   <details><summary>Answer</summary>Six districts in one season may not represent all 76 without sensors; a prior worth 20 bursts instead of 40 says so.</details>

## Carry forward

Lab 3.1 checks whether the Poisson part of this model fits the year-to-year variation.